In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, desc, sum, dayofweek, hour, date_format
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
import os
import sys

# ========================================
# SỬA LỖI Ở ĐÂY:
# Trỏ JAVA_HOME đến phiên bản OpenJDK 11
# ========================================
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'

# Thêm đường dẫn bin của Java vào PATH
os.environ['PATH'] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

In [ ]:
from pyspark.sql import SparkSession

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("BTL BigData - YouTube Analysis") \
    .getOrCreate()

print("✅ SparkSession đã khởi tạo.")


In [ ]:
# Đường dẫn HDFS tới thư mục data bạn vừa put
# Dùng localhost:9000 vì Docker đã "mở" cổng đó ra máy ảo
hdfs_path = "hdfs://localhost:9000/data/cleaned_data" 

df = spark.read.parquet(hdfs_path)

print("✅ Đọc dữ liệu sạch thành công!")
df.printSchema()
df.show(5)

In [ ]:
from pyspark.sql.functions import col, avg, count, desc, sum
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- 1. Dùng Spark để tính toán (Giữ nguyên) ---
print("\nĐang tính toán xu hướng theo Tháng...")
monthly_trends = df.groupBy("trending_year", "trending_month") \
    .agg(
        sum("view_count").alias("total_views"),
        count("video_id").alias("total_videos"),
        avg("like_ratio").alias("avg_like_ratio"),
        avg("engagement_rate").alias("avg_engagement")
    ) \
    .orderBy("trending_year", "trending_month")

print("Đã tính toán xong:")
monthly_trends.show()

# --- 2. Chuyển sang Pandas để vẽ (Giữ nguyên) ---
monthly_trends_pd = monthly_trends.toPandas()
# Tạo cột 'date' (ví dụ: '2020-08') để vẽ
monthly_trends_pd['date'] = pd.to_datetime(
    monthly_trends_pd['trending_year'].astype(str) + '-' + \
    monthly_trends_pd['trending_month'].astype(str)
)

# ================================================================
# --- 3. Vẽ biểu đồ (Code lại "dùng cái khác") ---
# ================================================================
print("\nĐang vẽ 2 biểu đồ xu hướng theo Tháng (đã gộp)...")

# Tạo 2 ô, 1 cho "Số lượng" (Volume), 1 cho "Chất lượng" (Quality)
fig, axes = plt.subplots(2, 1, figsize=(15, 12), sharex=True)

# --- Biểu đồ 1: Phân tích SỐ LƯỢNG (Dùng 2 trục Y) ---
# Trục Y bên trái (ax1) cho Total Views
color = 'tab:blue'
axes[0].set_title('Phân tích Số lượng (Volume) theo thời gian')
axes[0].set_xlabel('Ngày (Date)')
axes[0].set_ylabel('Total Views (triệu)', color=color)
axes[0].plot(monthly_trends_pd['date'], monthly_trends_pd['total_views'], color=color, marker='o', label='Total Views')
axes[0].tick_params(axis='y', labelcolor=color)

# Tạo Trục Y bên phải (ax1_twin) cho Total Videos
ax1_twin = axes[0].twinx()
color = 'tab:red'
ax1_twin.set_ylabel('Total Videos', color=color)
ax1_twin.plot(monthly_trends_pd['date'], monthly_trends_pd['total_videos'], color=color, marker='x', label='Total Videos')
ax1_twin.tick_params(axis='y', labelcolor=color)

# --- Biểu đồ 2: Phân tích CHẤT LƯỢNG (Dùng 1 trục Y, nhiều đường) ---
# (Vì 'like_ratio' và 'engagement' đều là %, chúng có thể chung 1 trục)

# 1. "Làm tan chảy" (Melt) DataFrame để Seaborn vẽ nhiều đường
quality_pd = monthly_trends_pd.melt(
    id_vars=['date'], 
    value_vars=['avg_like_ratio', 'avg_engagement'],
    var_name='Metric', 
    value_name='Percentage (%)'
)

# 2. Vẽ 2 đường trên 1 biểu đồ
sns.lineplot(data=quality_pd, x='date', y='Percentage (%)', hue='Metric', ax=axes[1], marker='o')
axes[1].set_title('Phân tích Chất lượng (Quality) theo thời gian')
axes[1].set_xlabel('Ngày (Date)')
axes[1].set_ylabel('Tỷ lệ (%)')
axes[1].grid(linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# --- 1. Dùng Spark để tính toán ---
print("\nĐang tính toán xu hướng theo Giờ & Ngày...")

# Thêm cột 'trending_dow' (Thứ) và 'publish_hour' (Giờ)
df_micro = df.withColumn("trending_dow", date_format(col("trending_date"), "E")) \
             .withColumn("publish_hour", hour(col("publish_time")))

# Tính toán theo Giờ
hourly_analysis = df_micro.groupBy("publish_hour") \
    .count().alias("total_videos") \
    .orderBy("publish_hour")

# Tính toán theo Thứ
dow_analysis = df_micro.groupBy("trending_dow") \
    .count().alias("total_videos")

# --- 2. Chuyển sang Pandas để vẽ ---
hourly_analysis_pd = hourly_analysis.toPandas()
dow_analysis_pd = dow_analysis.toPandas()

# Sắp xếp các ngày theo đúng thứ tự (Mon -> Sun)
days_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_analysis_pd['trending_dow'] = pd.Categorical(
    dow_analysis_pd['trending_dow'], categories=days_order, ordered=True
)
dow_analysis_pd = dow_analysis_pd.sort_values('trending_dow')

# --- 3. Vẽ biểu đồ ---
print("\nĐang vẽ biểu đồ xu hướng Giờ & Ngày...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6)) # 1 hàng, 2 cột

# Biểu đồ 1: Theo Giờ Xuất Bản
sns.lineplot(data=hourly_analysis_pd, x='publish_hour', y='count', marker='o', ax=ax1)
ax1.set_title('Số lượng video trending theo Giờ Xuất Bản (Publish Hour)')
ax1.set_xlabel('Giờ trong ngày (0-23h)')
ax1.set_ylabel('Total Videos')
ax1.grid(axis='x', linestyle='--', alpha=0.7)
ax1.set_xticks(range(0, 24))

# Biểu đồ 2: Theo Ngày Trending
sns.barplot(data=dow_analysis_pd, x='trending_dow', y='count', ax=ax2, palette='Blues_d')
ax2.set_title('Số lượng video trending theo Ngày trong Tuần')
ax2.set_xlabel('Ngày (Day of Week)')
ax2.set_ylabel('Total Videos')

plt.show()

In [ ]:
# --- 1. Dùng Spark để tính toán ---
print("\nĐang tính toán xu hướng Thể loại theo Thời gian...")

# Nhóm theo cả tháng và thể loại
category_time_analysis = df.groupBy("trending_month", "category_name") \
    .count() \
    .orderBy("trending_month", "count")

print("Đã tính toán xong:")
category_time_analysis.show()

# --- 2. Chuyển sang Pandas & Pivot để vẽ Heatmap ---
category_time_pd = category_time_analysis.toPandas()

# Pivot: Biến bảng 'dài' thành bảng 'rộng'
# Hàng là Category, Cột là Tháng, Giá trị là 'count'
category_pivot = category_time_pd.pivot_table(
    index='category_name', 
    columns='trending_month', 
    values='count'
).fillna(0) # Lấp đầy các tháng không có video = 0

# --- 3. Vẽ biểu đồ (Heatmap) ---
print("\nĐang vẽ biểu đồ Heatmap Thể loại theo Tháng...")
plt.figure(figsize=(15, 10))
sns.heatmap(
    category_pivot, 
    annot=True,     # Hiển thị con số
    fmt=".0f",      # Định dạng số nguyên
    cmap="viridis"  # Dải màu
)
plt.title('Heatmap: Số lượng video trending theo Thể loại theo Tháng')
plt.xlabel('Tháng (Month)')
plt.ylabel('Thể loại (Category)')
plt.show()

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import pandas as pd # Đảm bảo bạn đã import pandas

# (Các import khác như plt, sns...)

# --- 1. Phân tích Tương quan (Tương thích với Spark 3.2) ---
print("\nĐang tính toán Ma trận Tương quan...")

interaction_cols = ['view_count', 'likes', 'dislikes', 'comment_count', 'engagement_rate', 'like_ratio', 'days_to_trend']

# BƯỚC 1: Chuyển các cột số thành một cột vector duy nhất
# (Đây là yêu cầu bắt buộc của hàm Correlation)
assembler = VectorAssembler(
    inputCols=interaction_cols,
    outputCol="features",
    handleInvalid="skip" # Bỏ qua các dòng có giá trị NULL
)
df_features = assembler.transform(df).select("features")

# BƯỚC 2: Tính toán ma trận tương quan Pearson
# Lệnh .head() sẽ thực thi và trả về kết quả
corr_matrix_spark = Correlation.corr(df_features, "features").head()

# BƯỚC 3: Trích xuất ma trận (kết quả nằm ở chỉ số 0)
# Kết quả là một ma trận dày (DenseMatrix) của MLlib
corr_matrix = corr_matrix_spark[0] 

# Chuyển ma trận MLlib sang Numpy array, rồi sang Pandas DataFrame
corr_pd = pd.DataFrame(
    corr_matrix.toArray(), 
    columns=interaction_cols, 
    index=interaction_cols
)

print("Ma trận tương quan:")
print(corr_pd)

# --- 2. Vẽ biểu đồ Heatmap Tương quan ---
# (Phần code này của bạn không cần thay đổi)
print("\nĐang vẽ biểu đồ Heatmap Tương quan...")
plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_pd, 
    annot=True,     # Hiển thị số
    fmt=".2f",      # 2 chữ số thập phân
    cmap="vlag_r",  # Dải màu (Đỏ -> Trắng -> Xanh)
    vmin=-1, vmax=1
)
plt.title('Heatmap Tương quan giữa các Chỉ số Tương tác')
plt.show()